In [2]:
import matplotlib as plt
import sklearn
import numpy as np
import pandas as pd

In [3]:
def read(name):
    return pd.read_csv(f"dataset/{name}.csv", na_values=["\\N"])

def lap_milliseconds(t):
    if pd.isna(t) or ":" not in str(t):
        return np.nan
    minutes, seconds = str(t).split(":")
    return int(minutes) * 60 * 1000 + float(seconds) * 1000
    

In [4]:
races = read("races")
results = read("results")
qualifying = read("qualifying")
drivers = read("drivers")
circuits = read("circuits")
constructors_standings = read("constructor_standings")
constructors = read("constructors")
sprints = read("sprint_results")

# we will only choose races from 2014 until present
# because this period is hybrid engines period and much more usable
races = races[races["raceId"] > 899]
results = results[results["raceId"].isin(races["raceId"])]
qualifying = qualifying[qualifying["raceId"].isin(races["raceId"])]
constructors_standings = constructors_standings[constructors_standings["raceId"].isin(races["raceId"])]

# keeping needed columns
races = races[["raceId", "year", "round", "circuitId"]]
results = results[["raceId","driverId","constructorId","grid"]]
qualifying = qualifying[[ "raceId", "driverId", "constructorId", "q1", "q2", "q3" ]]
constructors_standings = constructors_standings[["raceId", "constructorId"]]

sprints = sprints.rename(columns={"position": "sprint_position", "grid": "sprint_grid"})
sprints = sprints[[ "raceId", "driverId", "constructorId", "sprint_grid", "sprint_position" ]]

In [6]:
df = (
    races
    .merge(results, on="raceId", how="left")
    .merge(
        qualifying,
        on=["raceId", "driverId", "constructorId"],
        how="left"
    )
    .merge(
        constructors_standings,
        on=["raceId", "constructorId"],
        how="left"
    )
    .merge(
        sprints,
        on=["raceId", "driverId", "constructorId"],
        how="left",
    )
)
df["q1"] = df["q1"].apply(lap_milliseconds)
df["q2"] = df["q2"].apply(lap_milliseconds)
df["q3"] = df["q3"].apply(lap_milliseconds)

df.dropna(subset=['driverId'], inplace = True)
df.dropna(subset=['constructorId'], inplace = True)
df.dropna(subset=['raceId'], inplace = True)
df.fillna({'grid': 0}, inplace = True)
df.fillna({'sprint_grid': 0, 'sprint_position': 0}, inplace = True)

df["q1_reached"] = df["q1"].notna().astype("int")
df["q2_reached"] = df["q2"].notna().astype("int")
df["q3_reached"] = df["q3"].notna().astype("int")

for c in ["q1", "q2", "q3"]:
    fastest = df.groupby("raceId")[c].transform("min")
    df[c] = (fastest / df[c]).fillna(0)   # pole positon is 1 and others are less than 1, if no time set then 0


df["driverId"] = df["driverId"].astype("int")
df["constructorId"] = df["constructorId"].astype("int")
df["raceId"] = df["raceId"].astype("int")
df["grid"] = df["grid"].astype("int")
df.head(30)


,raceId,year,round,circuitId,driverId,constructorId,grid,q1,q2,q3,sprint_grid,sprint_position,q1_reached,q2_reached,q3_reached
0,900,2014,1,1,3,131,3,0.980673,1.000000,0.996520,0.0,0.0,1,1,1
1,900,2014,1,1,825,1,4,0.998087,0.990479,0.985683,0.0,0.0,1,1,1
2,900,2014,1,1,18,1,10,0.993205,0.979193,0.000000,0.0,0.0,1,1,0
3,900,2014,1,1,4,6,5,0.993292,0.994738,0.984993,0.0,0.0,1,1,1
4,900,2014,1,1,822,3,15,0.990983,0.984709,0.963790,0.0,0.0,1,1,1
5,900,2014,1,1,807,10,7,0.966792,0.986552,0.983033,0.0,0.0,1,1,1
6,900,2014,1,1,8,6,11,0.981999,0.978659,0.000000,0.0,0.0,1,1,0
7,900,2014,1,1,818,5,6,0.970980,0.984737,0.984575,0.0,0.0,1,1,1
8,900,2014,1,1,826,5,8,0.967988,0.980188,0.970783,0.0,0.0,1,1,1
9,900,2014,1,1,815,10,16,0.964245,0.953128,0.000000,0.0,0.0,1,1,0
